<a href="https://colab.research.google.com/github/cursuri-inf/IPAP/blob/main/saptamana-04/L4_neuron_si_retea.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Caietul „De la un neuron la o rețea” (Laborator 4)

Pui la lucru bucla din laboratorul 3 pe două întrebări de clasificare. Întâi perechea grea, Adelie și Chinstrap, cu un singur neuron. Apoi întrebarea „mascul sau femelă?”, la care un neuron nu ajunge: completezi cele trei straturi ale unei rețele, alegi câți neuroni ascunși îi trebuie și o evaluezi o singură dată pe test. Nivelul de bază (secțiunile 0–4) se termină în laborator.

Pașii:

1. Deschizi caietul cu butonul „Open in Colab” de mai sus. La prima rulare, Colab avertizează că documentul nu a fost creat de Google: caietul vine din depozitul cursului, așa că alegi „Run anyway”.
2. Înainte să scrii ceva: File → Save a copy in Drive; lucrezi în copie.
3. Lucrezi în pereche cu un coleg din echipă. Pilotul rulează și scrie, navigatorul citește, prezice și verifică; schimbați rolurile la fiecare secțiune. Într-o echipă de 3, al treilea coleg este verificator: rulează în copia lui și compară rezultatele. Fiecare predă caietul lui.
4. Rulezi celulele în ordine, cu Shift+Enter. La fiecare „Exercițiul B…” te oprești și scrii răspunsul în celula „Răspunsul tău:” de sub el sau completezi codul, acolo unde vezi `...` sau `None`.
5. La final: Runtime → Restart session and run all (rularea de la zero), verifici că ultima celulă de cod afișează „Caietul a rulat până la capăt”, completezi „Declarația AI”, descarci caietul (File → Download → Download .ipynb) și îl încarci la activitatea L4 din Campus Virtual.

Tot caietul rulează pe procesor (CPU), în aproximativ un minut; nu ai nevoie de GPU. Extensiile E1–E3 sunt opționale, pentru nota 8–10.

Completează:

- Numele tău: …
- Colegul de pereche: …
- Echipa: …

## Harta caietului

Fiecare secțiune pune la lucru un capitol din suportul cursului 4.

| Secțiunea | Capitolul din suportul cursului 4 |
|---|---|
| 1. Adelie și Chinstrap cu un neuron | 2, Regresia logistică |
| 2. Mascul sau femelă: un neuron și o rețea | 4.1–4.2, Limita unei drepte; Straturi și ReLU |
| 3. Câți neuroni ascunși | 4.3, Rețeaua îndoaie granița |
| 4. Testul, o singură dată | cursul 2, împărțirea datelor |
| Extensia E1 | 3.3, Trei clase: softmax |
| Extensia E2 | 5, Propagarea înapoi |

## 0. Pornire

Încarci bibliotecile și datele (cei 342 de pinguini cu toate măsurătorile) și pregătești bucla de antrenare. Funcția `pas` este cea pe care ai completat-o la laboratorul 3; aici este gata scrisă. Dacă adresa datelor nu răspunde, descarci fișierul `penguins.csv` din Campus Virtual (Cursul 2, „Datele cu pinguini”), îl încarci în panoul Files din bara din stânga și rulezi din nou celula cu datele.

In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.optim import SGD
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

T_START = time.time()  # pentru durata totală, afișată la final

print("Python ", sys.version.split()[0])
print("PyTorch", torch.__version__)


def procent(x):
    """0.9394 -> '93,9%' (cu virgulă zecimală)."""
    return f"{100 * x:.1f}%".replace(".", ",")


# culorile graficelor cursului
TEXT, SECUNDAR, ACCENT, SERIE2 = "#161616", "#4A4A4A", "#9B1B30", "#4E6375"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": False,
                     "text.color": TEXT, "axes.labelcolor": SECUNDAR,
                     "xtick.color": SECUNDAR, "ytick.color": SECUNDAR})

In [ ]:
URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
masuri = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

sursa = "penguins.csv" if os.path.exists("penguins.csv") else URL
df = pd.read_csv(sursa).dropna(subset=masuri)  # 342 de pinguini cu toate măsurătorile
print("Pinguini:", len(df))

In [ ]:
def pas(model, pierdere_f, opt, zb, yb):
    """Un pas al buclei, pe un lot (funcția din laboratorul 3)."""
    predictie = model(zb)
    pierdere = pierdere_f(predictie, yb)
    opt.zero_grad()
    pierdere.backward()
    opt.step()


def antreneaza(model, pierdere_f, z, y, rata, epoci):
    """Bucla: la fiecare epocă, câte un pas pe fiecare lot de 32."""
    opt = SGD(model.parameters(), lr=rata)
    loturi = DataLoader(TensorDataset(z, y), batch_size=32, shuffle=True)
    for epoca in range(epoci):
        for zb, yb in loturi:
            pas(model, pierdere_f, opt, zb, yb)


def acuratete(model, z, y):
    """Ce parte din exemple primesc clasa corectă (clasa 1 când scorul este pozitiv)."""
    with torch.no_grad():
        return ((model(z) > 0).float() == y).float().mean().item()

## 1. Adelie și Chinstrap cu un neuron

La curs, pinguinii Gentoo s-au despărțit de ceilalți printr-o dreaptă, fără nicio greșeală. Perechea grea este Adelie și Chinstrap: au aproape aceeași masă și aripi asemănătoare. Folosești împărțirea pentru clasificare din cursul 2 (70% antrenare, 15% validare, 15% test, stratificat pe specii) și păstrezi doar cele două specii. Eticheta este 1 pentru Chinstrap și 0 pentru Adelie.

Referința simplă: cineva care răspunde mereu cu specia mai frecventă, Adelie.

In [ ]:
X_rest, X_test, s_rest, s_test = train_test_split(
    df[masuri], df["species"], test_size=0.15,
    stratify=df["species"], random_state=0)
X_ant, X_val, s_ant, s_val = train_test_split(
    X_rest, s_rest, test_size=0.15 / 0.85,
    stratify=s_rest, random_state=0)


def pereche(X, s):
    """Doar Adelie și Chinstrap; eticheta 1 pentru Chinstrap, 0 pentru Adelie."""
    pastram = s.isin(["Adelie", "Chinstrap"]).values
    eticheta = torch.tensor((s[pastram] == "Chinstrap").values).float().reshape(-1, 1)
    return X[pastram], eticheta


A_ant, t_ant = pereche(X_ant, s_ant)
A_val, t_val = pereche(X_val, s_val)
print("antrenare", len(A_ant), "| validare", len(A_val))

REFERINTA_AC = 1 - t_val.mean().item()  # mereu Adelie
print("Chinstrap pe validare:", procent(t_val.mean().item()),
      "| referința (mereu Adelie):", procent(REFERINTA_AC))

Un neuron pe două măsurători, standardizate pe antrenare, cu pierderea `nn.BCEWithLogitsLoss` (cursul 4, §2–§3). Antrenezi două modele: unul pe lungimea și adâncimea ciocului, altul pe lungimea aripii și adâncimea ciocului.

In [ ]:
def intrari(X, col):
    """Coloanele alese, standardizate cu media și abaterea de la antrenare."""
    m, s = A_ant[col].mean(), A_ant[col].std()
    return torch.tensor(((X[col] - m) / s).values).float()


def neuron_pe(col):
    torch.manual_seed(0)
    neuron = nn.Linear(len(col), 1)  # scorul = w1 × prima + w2 × a doua + b
    antreneaza(neuron, nn.BCEWithLogitsLoss(), intrari(A_ant, col), t_ant, rata=0.5, epoci=50)
    return neuron


CIOC = ["bill_length_mm", "bill_depth_mm"]
ARIPA = ["flipper_length_mm", "bill_depth_mm"]
neuron_cioc, neuron_aripa = neuron_pe(CIOC), neuron_pe(ARIPA)
ACC_CIOC = acuratete(neuron_cioc, intrari(A_val, CIOC), t_val)
ACC_ARIPA = acuratete(neuron_aripa, intrari(A_val, ARIPA), t_val)
print("ciocul (lungime, adâncime):  validare", procent(ACC_CIOC))
print("aripa și adâncimea ciocului: validare", procent(ACC_ARIPA))
print("referința (mereu Adelie):    validare", procent(REFERINTA_AC))
if ACC_CIOC < REFERINTA_AC:
    print("\nAtenție: modelul cu ciocul este sub referință. Verifică în pas() ordinea din pierdere_f:",
          "întâi predicția, apoi eticheta (yb).")

In [ ]:
NUME_COL = {"bill_length_mm": "lungimea ciocului (mm)", "bill_depth_mm": "adâncimea ciocului (mm)",
            "flipper_length_mm": "lungimea aripii (mm)", "body_mass_g": "masa corporală (g)"}


def granita(ax, model, col, X, eticheta, nume, medie, abatere, titlu):
    """Punctele de antrenare și granița de decizie a modelului (scorul 0), în unitățile datelor."""
    for et, n, culoare in ((0, nume[0], SECUNDAR), (1, nume[1], ACCENT)):
        k = (eticheta[:, 0] == et).numpy()
        ax.scatter(X[col[0]].values[k], X[col[1]].values[k], s=14, color=culoare, alpha=0.75, label=n)
    gx = np.linspace(X[col[0]].min(), X[col[0]].max(), 200)
    gy = np.linspace(X[col[1]].min(), X[col[1]].max(), 200)
    XX, YY = np.meshgrid(gx, gy)
    z = np.c_[(XX.ravel() - medie[0]) / abatere[0], (YY.ravel() - medie[1]) / abatere[1]]
    with torch.no_grad():
        S = model(torch.tensor(z).float()).numpy().reshape(XX.shape)
    ax.contour(XX, YY, S, levels=[0], colors=[TEXT], linewidths=1.5, linestyles="--")
    ax.set_xlabel(NUME_COL[col[0]])
    ax.set_ylabel(NUME_COL[col[1]])
    ax.set_title(titlu, loc="left")


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for ax, model, col, titlu in ((ax1, neuron_cioc, CIOC, "ciocul"), (ax2, neuron_aripa, ARIPA, "aripa și adâncimea ciocului")):
    granita(ax, model, col, A_ant, t_ant, ("Adelie", "Chinstrap"),
            A_ant[col].mean().values, A_ant[col].std().values, titlu)
ax1.legend(frameon=False)
plt.tight_layout()
plt.show()

Linia punctată este granița de decizie: scorul este 0 pe ea. De o parte, modelul spune Chinstrap; de cealaltă, Adelie.

### Exercițiul B1 (interpretează)

1. Ce măsurătoare deosebește Adelie de Chinstrap? Cum se vede asta pe graficul din stânga?
2. Modelul cu aripa și adâncimea ciocului are acuratețea referinței. A învățat ceva util? De ce nu poate face mai mult cu aceste două măsurători? Uită-te la graficul din dreapta.

Răspunsul tău:

## 2. Mascul sau femelă: un neuron și o rețea

La curs, la întrebarea „mascul sau femelă?”, modelele au primit masa corporală și lungimea ciocului. Aici primesc lungimea aripii și lungimea ciocului. Folosești cei 333 de pinguini pentru care se știe, cu aceeași împărțire 70/15/15, stratificată pe specii. Eticheta este 1 pentru mascul și 0 pentru femelă (în tabel, coloana `sex`).

In [ ]:
mf = df.dropna(subset=["sex"])  # 333 de pinguini
M_rest, M_test = train_test_split(mf, test_size=0.15, stratify=mf["species"], random_state=0)
M_ant, M_val = train_test_split(M_rest, test_size=0.15 / 0.85, stratify=M_rest["species"], random_state=0)

COL_MF = ["flipper_length_mm", "bill_length_mm"]
MED, AB = M_ant[COL_MF].mean(), M_ant[COL_MF].std()


def standard(t):
    """Aripa și ciocul, standardizate cu media și abaterea de la antrenare."""
    return torch.tensor(((t[COL_MF] - MED) / AB).values).float()


def mascul(t):
    """1 pentru mascul, 0 pentru femelă."""
    return torch.tensor((t["sex"] == "MALE").values).float().reshape(-1, 1)


zm_ant, zm_val = standard(M_ant), standard(M_val)
ym_ant, ym_val = mascul(M_ant), mascul(M_val)
REFERINTA_MF = 1 - ym_val.mean().item()  # mereu femelă
print("antrenare", len(M_ant), "| validare", len(M_val), "| test", len(M_test))
print("referința (mereu femelă) pe validare:", procent(REFERINTA_MF))

In [ ]:
torch.manual_seed(0)
neuron_mf = nn.Linear(2, 1)
antreneaza(neuron_mf, nn.BCEWithLogitsLoss(), zm_ant, ym_ant, rata=0.1, epoci=300)
NEURON_ANT, NEURON_VAL = acuratete(neuron_mf, zm_ant, ym_ant), acuratete(neuron_mf, zm_val, ym_val)
print("un neuron: antrenare", procent(NEURON_ANT), "| validare", procent(NEURON_VAL))

### Exercițiul B2 (completează)

În locul celor trei `...` scrii straturile rețelei din curs (§4.2): un strat ascuns cu 2 intrări și 16 neuroni, funcția de activare ReLU, apoi neuronul de ieșire, cu 16 intrări și o ieșire. Rândul cu `torch.manual_seed(0)` rămâne primul, ca rezultatele să se poată compara. Celula de după antrenează rețeaua cu aceeași buclă, aceeași pierdere, aceeași rată și același număr de epoci ca neuronul și îți spune dacă straturile sunt corecte.

In [ ]:
# Exercițiul B2: înlocuiește fiecare ... cu stratul care lipsește
torch.manual_seed(0)
straturi = [
    ...,   # stratul ascuns: 2 intrări, 16 neuroni
    ...,   # funcția de activare
    ...,   # neuronul de ieșire: 16 intrări, o ieșire
]

In [ ]:
B2_CORECT = False
retea = neuron_mf  # până la o rețea corectă, graficul de mai jos arată tot neuronul
if any(s is ... for s in straturi):
    print("B2: nu încă. Completează cele trei straturi,",
          "apoi rulează din nou celula cu straturile și pe aceasta.")
elif not all(isinstance(s, nn.Module) for s in straturi):
    print("B2: nu încă. Fiecare strat se scrie cu paranteze: nn.ReLU(), nu nn.ReLU.")
elif isinstance(straturi[-1], nn.Sigmoid):
    print("B2: nu încă. Ultimul strat este neuronul de ieșire, nn.Linear(16, 1);",
          "pierderea BCEWithLogitsLoss aplică singură sigmoida (§3).")
else:
    # de la zero la fiecare rulare: aceleași valori de pornire ca în celula cu straturile
    torch.manual_seed(0)
    for s in straturi:
        if hasattr(s, "reset_parameters"):
            s.reset_parameters()
    try:
        r = nn.Sequential(*straturi)
        antreneaza(r, nn.BCEWithLogitsLoss(), zm_ant, ym_ant, rata=0.1, epoci=300)
    except (RuntimeError, ValueError) as e:
        print("B2: nu încă. Formele straturilor nu se potrivesc:", str(e).splitlines()[0])
        print("Primul strat primește 2 intrări; al doilea primește atâtea intrări",
              "câți neuroni are primul și dă o singură ieșire.")
    else:
        retea = r
        RETEA_ANT, RETEA_VAL = acuratete(r, zm_ant, ym_ant), acuratete(r, zm_val, ym_val)
        print("rețeaua:  antrenare", procent(RETEA_ANT), "| validare", procent(RETEA_VAL))
        print("parametri:", sum(p.numel() for p in r.parameters()))
        if any(isinstance(s, nn.ReLU) for s in straturi):
            if RETEA_VAL >= 0.75:
                B2_CORECT = True
                print("\nB2: corect. Rețeaua trece clar de neuron:", procent(RETEA_VAL),
                      "față de", procent(NEURON_VAL), "pe validare.")
            else:
                print("\nB2: nu încă. Rețeaua nu trece clar de neuron.",
                      "Verifică numărul de neuroni (16) și ordinea straturilor.")
        elif any(isinstance(s, (nn.Sigmoid, nn.Tanh)) for s in straturi):
            print("\nB2: nu încă. Exercițiul cere ReLU, ca la curs (§4.2).")
        else:
            print("\nB2: nu încă. Rețeaua nu are funcție de activare între straturi:",
                  "două straturi liniare fac tot o dreaptă (§4.2).")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for ax, model, titlu in ((ax1, neuron_mf, "un neuron"), (ax2, retea, "rețeaua")):
    granita(ax, model, COL_MF, M_ant, ym_ant, ("femele", "masculi"), MED.values, AB.values, titlu)
ax1.legend(frameon=False)
plt.tight_layout()
plt.show()

## 3. Câți neuroni ascunși

Funcția `retea_cu` construiește și antrenează o rețea ca a ta, cu numărul de neuroni ascunși dat, și întoarce acuratețea pe antrenare și pe validare.

### Exercițiul B3 (modifică și notează)

1. Rulează celula și completează tabelul de mai jos pentru 2, 4, 16 și 64 de neuroni ascunși.
2. Adaugă în listă un număr al tău (de exemplu 8 sau 32), rulează din nou și completează ultimul rând.
3. Câți neuroni ascunși alegi și de ce? (2–3 fraze; alegerea se face pe validare. Ține cont că setul de validare are 50 de pinguini: un pinguin înseamnă 2%.) Scrie numărul ales în celula de la secțiunea 4.

In [ ]:
def retea_cu(ascunsi):
    torch.manual_seed(0)
    r = nn.Sequential(nn.Linear(2, ascunsi), nn.ReLU(), nn.Linear(ascunsi, 1))
    antreneaza(r, nn.BCEWithLogitsLoss(), zm_ant, ym_ant, rata=0.1, epoci=300)
    return r, acuratete(r, zm_ant, ym_ant), acuratete(r, zm_val, ym_val)


for ascunsi in (2, 4, 16, 64):  # Exercițiul B3: adaugă un număr al tău
    _, a, v = retea_cu(ascunsi)
    print(f"{ascunsi:3} neuroni ascunși: antrenare {procent(a):>6} | validare {procent(v):>6}")

Răspunsul tău:

| Neuroni ascunși | Antrenare | Validare |
|---|---|---|
| 2 | | |
| 4 | | |
| 16 | | |
| 64 | | |
| numărul tău: … | | |

Ce număr alegi și de ce:

## 4. Testul, o singură dată

Modelul se alege pe validare (secțiunea 3) și se evaluează o singură dată pe test, la final, ca în cursul 2. Scrie în `ASCUNSI_ALES` numărul de neuroni ales la B3 și rulează celula: ea antrenează rețeaua aleasă și o evaluează pe cei 50 de pinguini de test, alături de neuron și de referință.

### Exercițiul B4 (interpretează)

1. Cât are rețeaua aleasă pe test, față de validare și față de neuron? De ce pot diferi cifra de pe validare și cea de pe test?
2. De ce evaluăm pe test o singură dată, abia după ce am ales modelul? Ce s-ar întâmpla dacă am alege numărul de neuroni după test?
3. De ce granița dintre masculi și femele trebuie să fie îndoită? Gândește-te la cele trei specii.

In [ ]:
ASCUNSI_ALES = None  # Exercițiul B4: numărul de neuroni ascunși ales la B3

if ASCUNSI_ALES is None:
    print("B4: scrie întâi în ASCUNSI_ALES numărul de neuroni ales la B3.")
    ACC_TEST = None
else:
    ales, _, _ = retea_cu(ASCUNSI_ALES)
    zm_test, ym_test = standard(M_test), mascul(M_test)
    ACC_TEST = acuratete(ales, zm_test, ym_test)
    print(f"rețeaua cu {ASCUNSI_ALES} neuroni ascunși, pe test:", procent(ACC_TEST))
    print("un neuron, pe test:", procent(acuratete(neuron_mf, zm_test, ym_test)))
    print("referința (mereu femelă), pe test:", procent(1 - ym_test.mean().item()))

Răspunsul tău:

## Extensii (opționale, pentru nota 8–10)

Alegi cel puțin una. Fiecare are un comutator: îl pui pe `True`, rulezi celula și scrii, în celula „Comentariul tău” de sub ea, ce ai schimbat, ce rezultat ai obținut și cum îl explici. Extensia pe care o predai rămâne cu comutatorul pe `True`, ca rezultatele ei să se vadă în caiet. Toate rulează pe procesor, în câteva secunde. Rulează extensiile după ce nivelul de bază este gata.

### Extensia E1: softmax pe trei specii

Toate cele trei specii deodată, ca în cursul 4 (§3.3): un strat cu trei ieșiri, câte un scor pe specie, pe trei măsurători (ciocul și aripa), și pierderea `nn.CrossEntropyLoss`, care face softmax și −ln într-un singur pas. Compară acuratețea pe validare cu referința (specia cea mai frecventă) și citește matricea de confuzie: care specii se confundă între ele?

In [ ]:
RULEAZA_E1 = False  # True: rulezi extensia

if RULEAZA_E1:
    specii = ["Adelie", "Chinstrap", "Gentoo"]
    col3 = masuri[:3]
    m3, s3 = X_ant[col3].mean(), X_ant[col3].std()
    z3_ant = torch.tensor(((X_ant[col3] - m3) / s3).values).float()
    z3_val = torch.tensor(((X_val[col3] - m3) / s3).values).float()
    c_ant = torch.tensor(s_ant.map(specii.index).values)  # 0, 1, 2
    c_val = torch.tensor(s_val.map(specii.index).values)
    torch.manual_seed(0)
    strat = nn.Linear(3, 3)  # un scor pe specie
    antreneaza(strat, nn.CrossEntropyLoss(), z3_ant, c_ant, rata=0.5, epoci=50)
    with torch.no_grad():
        alese = strat(z3_val).argmax(dim=1)
    print("acuratețea pe validare:  ", procent((alese == c_val).float().mean().item()))
    print("referința (mereu Adelie):", procent((c_val == 0).float().mean().item()))
    M = torch.zeros(3, 3, dtype=torch.int64)
    for real, prezis in zip(c_val, alese):
        M[real, prezis] += 1
    print("matricea de confuzie (rândurile: specia reală; coloanele: specia prezisă)")
    for i, sp in enumerate(specii):
        print(f"  {sp:10}", M[i].tolist())
else:
    print("E1 nu rulează (RULEAZA_E1 = False).")

Comentariul tău (E1):

### Extensia E2: micrograd, propagarea înapoi de la zero

O versiune foarte mică a ideii din micrograd (A. Karpathy): o clasă `Valoare` care ține minte cum a fost calculat un număr și își calculează singură gradientul, ca `backward()` din PyTorch (cursul 4, §5). Adunarea este scrisă; la înmulțire completezi cele două rânduri cu `...` din `_inapoi`, cu regula lanțului: panta lui a × b după a este b, iar după b este a. Apoi calculezi pierderea pătratică a unui neuron cu două intrări și compari gradienții cu cei dați de PyTorch. Ce observi și de ce?

In [ ]:
RULEAZA_E2 = False  # True: rulezi extensia

if RULEAZA_E2:
    class Valoare:
        """Un număr care ține minte din ce a fost calculat și își calculează gradientul."""
        def __init__(self, valoare, copii=()):
            self.valoare, self.grad = valoare, 0.0
            self._copii, self._inapoi = copii, lambda: None

        def __add__(self, alta):
            rezultat = Valoare(self.valoare + alta.valoare, (self, alta))
            def _inapoi():
                self.grad += rezultat.grad  # panta lui a + b după a este 1
                alta.grad += rezultat.grad
            rezultat._inapoi = _inapoi
            return rezultat

        def __mul__(self, alta):
            rezultat = Valoare(self.valoare * alta.valoare, (self, alta))
            def _inapoi():
                self.grad += ... * rezultat.grad  # completează: panta lui a × b după a
                alta.grad += ... * rezultat.grad  # completează: panta lui a × b după b
            rezultat._inapoi = _inapoi
            return rezultat

        def backward(self):
            ordine, vazute = [], set()
            def parcurge(v):  # de la intrări spre rezultat
                if v not in vazute:
                    vazute.add(v)
                    for c in v._copii:
                        parcurge(c)
                    ordine.append(v)
            parcurge(self)
            self.grad = 1.0
            for v in reversed(ordine):  # înapoi: de la rezultat spre intrări
                v._inapoi()

    x1, x2, y = Valoare(0.5), Valoare(-1.2), Valoare(1.0)
    w1, w2, b = Valoare(0.3), Valoare(-0.7), Valoare(0.1)
    scor = w1 * x1 + w2 * x2 + b
    eroare = scor + Valoare(-1.0) * y  # scorul − eticheta
    pierdere = eroare * eroare
    try:
        pierdere.backward()
        print("Valoare:", round(w1.grad, 4), round(w2.grad, 4), round(b.grad, 4))
    except TypeError:
        print("E2: completează întâi cele două rânduri cu ... din __mul__.")

    w = torch.tensor([0.3, -0.7], requires_grad=True)
    bt = torch.tensor(0.1, requires_grad=True)
    p = ((w * torch.tensor([0.5, -1.2])).sum() + bt - 1.0) ** 2
    p.backward()
    print("PyTorch:", [round(g, 4) for g in w.grad.tolist()], round(bt.grad.item(), 4))
else:
    print("E2 nu rulează (RULEAZA_E2 = False).")

Comentariul tău (E2):

### Extensia E3: datele echipei

Aceeași comparație, un neuron față de o rețea, pe un tabel ales de echipa ta: un fișier CSV cu o coloană de prezis care are exact două valori (ținta) și 1–3 coloane numerice de intrare. Folosești doar date publice (de exemplu de pe data.gov.ro sau Kaggle) sau date generate de voi; nu folosești date personale și nici date interne de la locul de muncă.

Încarci fișierul în panoul Files din bara din stânga sau pui `INCARCA_CSV = True` și îl alegi. Completezi numele fișierului, ținta și intrările. Codul împarte datele 70/15/15, standardizează intrările pe antrenare, antrenează un neuron și o rețea cu 16 neuroni ascunși și compară acuratețea pe validare cu referința simplă (clasa mai frecventă). După ce ai încărcat fișierul, pui înapoi `INCARCA_CSV = False`: altfel, la „Run all”, caietul se oprește și așteaptă un fișier.

In [ ]:
INCARCA_CSV = False             # True: alegi fișierul de pe calculator (doar în Colab)
FISIER_E3 = "datele_noastre.csv"
TINTA = "..."                   # coloana de prezis, cu exact două valori
INTRARI = ["..."]               # 1–3 coloane numerice
RATA_E3 = 0.1

if INCARCA_CSV:
    from google.colab import files  # există doar în Colab
    files.upload()

if os.path.exists(FISIER_E3):
    d3 = pd.read_csv(FISIER_E3).dropna(subset=INTRARI + [TINTA])
    valori = sorted(d3[TINTA].unique())
    assert len(valori) == 2, f"Ținta are {len(valori)} valori; trebuie să aibă exact două."
    y3 = torch.tensor((d3[TINTA] == valori[1]).values).float().reshape(-1, 1)  # 1 pentru a doua valoare
    idx = np.arange(len(d3))
    i_rest, i_test = train_test_split(idx, test_size=0.15, random_state=0)
    i_ant, i_val = train_test_split(i_rest, test_size=0.15 / 0.85, random_state=0)
    X3 = d3[INTRARI].astype(float).values
    m3, s3 = X3[i_ant].mean(axis=0), X3[i_ant].std(axis=0)
    z3 = torch.tensor((X3 - m3) / s3).float()
    k = len(INTRARI)
    torch.manual_seed(0)
    neuron3 = nn.Linear(k, 1)
    antreneaza(neuron3, nn.BCEWithLogitsLoss(), z3[i_ant], y3[i_ant], rata=RATA_E3, epoci=300)
    torch.manual_seed(0)
    retea3 = nn.Sequential(nn.Linear(k, 16), nn.ReLU(), nn.Linear(16, 1))
    antreneaza(retea3, nn.BCEWithLogitsLoss(), z3[i_ant], y3[i_ant], rata=RATA_E3, epoci=300)
    mai_frecventa = float(y3[i_ant].mean().item() >= 0.5)
    print("rânduri:", len(d3), "| antrenare", len(i_ant), "| validare", len(i_val), "| test", len(i_test))
    print("un neuron, validare:", procent(acuratete(neuron3, z3[i_val], y3[i_val])))
    print("rețeaua, validare:  ", procent(acuratete(retea3, z3[i_val], y3[i_val])))
    print("referința (clasa mai frecventă):", procent((y3[i_val] == mai_frecventa).float().mean().item()))
else:
    print(f"E3 nu rulează: fișierul {FISIER_E3} nu există.")

Comentariul tău (E3): ce date ați ales și de unde, ce ați prezis, cum se compară neuronul cu rețeaua și cu referința simplă și ce înseamnă asta pentru forma graniței.

## Verificarea finală

Celula de mai jos are sens după Runtime → Restart session and run all: dacă afișează mesajul, caietul rulează de la început până la sfârșit fără erori.

In [ ]:
durata = time.time() - T_START
print(f"Caietul a rulat până la capăt (durata: {durata:.0f} s).")
print("Adelie și Chinstrap, ciocul: validare", procent(ACC_CIOC), "| referința", procent(REFERINTA_AC))
print("Mascul sau femelă, un neuron: validare", procent(NEURON_VAL), "| referința", procent(REFERINTA_MF))
if not B2_CORECT:
    print("Atenție: rețeaua din B2 nu este încă corectă.")
if ACC_TEST is None:
    print("Atenție: B4 nu are încă numărul de neuroni ales (ASCUNSI_ALES).")

## Ce ai învățat

- Un neuron clasifică cu o dreaptă: dacă măsurătorile alese despart clasele (ciocul, la Adelie și Chinstrap), ajunge; dacă nu (aripa), nu învață nimic util, oricât l-ai antrena.
- O rețea cu un strat ascuns și ReLU îndoaie granița și poate urma zone diferite ale datelor, de exemplu speciile, la întrebarea „mascul sau femelă?”.
- Numărul de neuroni ascunși se alege pe validare; testul se folosește o singură dată, la final.
- Bucla de antrenare și funcția `pas` sunt aceleași pentru un neuron și pentru o rețea.

## Legătura cu cursul 5

La cursul 5 urmărești antrenarea pe parcurs: curbele pierderii pe antrenare și pe validare, epocă cu epocă, și optimizatori mai buni decât SGD, care aleg singuri mărimea pasului pentru fiecare parametru. Bucla rămâne aceeași; se schimbă optimizatorul și ce măsori la fiecare epocă.

## Declarație AI

Completezi la fiecare caiet predat, la persoana întâi:

- Instrumentul AI: …
- Pentru ce l-am folosit: …
- Ce am verificat singur (și cum): …

Dacă nu ai folosit niciun instrument AI, înlocuiești cele trei rânduri cu: Nu am folosit asistenți AI.